# Benchmark Datacenter GPU — `ordpy-gpu` (Seção 5.6)
1. Vá em **Ambiente de execução -> Alterar tipo de ambiente de execução -> T4 GPU** (ou L4/A100).
2. Execute todas as células (**Ctrl+F9**).
3. Copie o bloco JSON final e cole no chat.

In [1]:
!pip install -q git+https://github.com/fabio-linhares/ordpy-gpu.git ordpy
!nvidia-smi --query-gpu=name,memory.total,clocks.max.sm,compute_cap --format=csv,noheader

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
NVIDIA A100-SXM4-80GB, 81920 MiB, 1410 MHz, 8.0


In [2]:
import json, time
import numpy as np
import cupy as cp
import ordpy
from ordpy_gpu import complexity_entropy_batch

props = cp.cuda.runtime.getDeviceProperties(0)
gpu_name = props['name'].decode()
sm_count = int(props['multiProcessorCount'])
vram_gb = round(props['totalGlobalMem'] / (1024**3), 2)
cc = f"{props['major']}.{props['minor']}"

CUDA_CORES_MAP = {
    'T4': 2560, 'L4': 7424, 'A100': 6912, 'V100': 5120, 'P100': 3584
}
cuda_cores = next((v for k, v in CUDA_CORES_MAP.items() if k in gpu_name), sm_count * 64)

rng = np.random.default_rng(42)
B = 13

def bench_gpu(Nr, Nc, dx, dy, reps=15):
    img = rng.integers(0, 256, (B, Nr, Nc), dtype=np.uint8)
    # Warmup
    complexity_entropy_batch(img, dx=dx, dy=dy)
    cp.cuda.Device(0).synchronize()
    times_ms = []
    for _ in range(reps):
        t0 = time.perf_counter()
        complexity_entropy_batch(img, dx=dx, dy=dy)
        cp.cuda.Device(0).synchronize()
        times_ms.append((time.perf_counter() - t0) * 1000.0)
    arr = np.array(times_ms)
    mean_ms = float(arr.mean())
    ic95_ms = float(1.96 * arr.std(ddof=1) / np.sqrt(reps))
    windows = B * (Nr - dx + 1) * (Nc - dy + 1)
    throughput_gws = (windows / (mean_ms / 1000.0)) / 1e9
    return {
        'res': f'{Nr}x{Nc}', 'dx': dx, 'dy': dy, 'windows': int(windows),
        'mean_ms': round(mean_ms, 4), 'ic95_ms': round(ic95_ms, 4),
        'g_win_per_s': round(throughput_gws, 3)
    }

# 1. Paridade numérica em 800x800 (2x2)
img_eq = rng.integers(0, 256, (B, 800, 800), dtype=np.uint8)
H_gpu, C_gpu = complexity_entropy_batch(img_eq, dx=2, dy=2)
H_cpu, C_cpu = [], []
for c in range(B):
    hc = ordpy.complexity_entropy(img_eq[c], dx=2, dy=2)
    H_cpu.append(hc[0]); C_cpu.append(hc[1])
max_dh = float(np.max(np.abs(np.array(H_gpu) - np.array(H_cpu))))
max_dc = float(np.max(np.abs(np.array(C_gpu) - np.array(C_cpu))))

# 2. Benchmark 2x2 e 3x3 (inclui 4096x4096 para W=3 se VRAM suportar)
w2_runs = [bench_gpu(r, r, 2, 2) for r in [128, 256, 512, 800, 1024, 2048, 4096]]
w3_runs = []
for r in [800, 1024, 2048, 4096]:
    try:
        w3_runs.append(bench_gpu(r, r, 3, 3))
    except Exception as e:
        w3_runs.append({'res': f'{r}x{r}', 'error': str(e)})

ref_800 = next(x for x in w2_runs if x['res'] == '800x800')
per_core_win_s = (ref_800['g_win_per_s'] * 1e9) / cuda_cores

report = {
    'gpu_name': gpu_name, 'cc': cc, 'vram_gb': vram_gb,
    'sm_count': sm_count, 'cuda_cores': cuda_cores,
    'max_delta_H': max_dh, 'max_delta_C': max_dc,
    'w2_800x800_win_per_s_per_core': round(per_core_win_s, 1),
    'w2_runs': w2_runs,
    'w3_runs': w3_runs
}
print('=== COPIE O JSON ABAIXO ===')
print(json.dumps(report, indent=2))


=== COPIE O JSON ABAIXO ===
{
  "gpu_name": "NVIDIA A100-SXM4-80GB",
  "cc": "8.0",
  "vram_gb": 79.25,
  "sm_count": 108,
  "cuda_cores": 6912,
  "max_delta_H": 3.3306690738754696e-16,
  "max_delta_C": 1.4663732738912777e-15,
  "w2_800x800_win_per_s_per_core": 708767.4,
  "w2_runs": [
    {
      "res": "128x128",
      "dx": 2,
      "dy": 2,
      "windows": 209677,
      "mean_ms": 0.5495,
      "ic95_ms": 0.0166,
      "g_win_per_s": 0.382
    },
    {
      "res": "256x256",
      "dx": 2,
      "dy": 2,
      "windows": 845325,
      "mean_ms": 0.6272,
      "ic95_ms": 0.0093,
      "g_win_per_s": 1.348
    },
    {
      "res": "512x512",
      "dx": 2,
      "dy": 2,
      "windows": 3394573,
      "mean_ms": 0.8476,
      "ic95_ms": 0.0165,
      "g_win_per_s": 4.005
    },
    {
      "res": "800x800",
      "dx": 2,
      "dy": 2,
      "windows": 8299213,
      "mean_ms": 1.6939,
      "ic95_ms": 0.0599,
      "g_win_per_s": 4.899
    },
    {
      "res": "1024x1024",
   